# Prepare the clinical fine-tuning dataset

Run this notebook **before prepared_baseline_evaluation.ipynb and before training**. It saves a versioned, cleaned Hugging Face DatasetDict plus JSONL exports, tokenized assistant-only labels, source-row identities, rejection/repair logs, and a manifest. No model weights, inference, training, or paid API calls are used. CPU is sufficient.

Defaults: first 1,500 source training rows; 10 reproducibly selected validation cases; maximum complete training sequence 4,096 tokens. These are candidate counts: rejected/duplicate/overlength training rows reduce the final count. Evaluation selection draws from all eligible validation rows. Existing raw artifacts are preserved.

Repairs only reserialize valid JSON or JSON containing unescaped control characters (and optional Markdown fences). Broken syntax, duplicate keys, unexpected top-level keys, and nonstandard constants are rejected. This validates JSON structure, not clinical correctness. Exact normalized-note duplicates are removed, including across train/evaluation; near duplicates and patient-level leakage are not detected.


In [1]:
%pip install --progress-bar on "transformers==4.57.6" "datasets>=3,<5" huggingface_hub pandas


In [2]:
import os, json, hashlib, random, re, time, importlib.metadata
from pathlib import Path
from getpass import getpass
from collections import Counter
import pandas as pd
from huggingface_hub import HfApi, get_token
from transformers import AutoTokenizer

def digest(value):
    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True).encode()).hexdigest()

def read_token():
    token = os.environ.get("HF_TOKEN") or get_token()
    if not token:
        print("Enter your Hugging Face token in the VS Code input prompt.", flush=True)
        token = getpass("Hugging Face read token: ").strip()
    if not token:
        raise RuntimeError("A Hugging Face token with accepted Gemma access is required.")
    os.environ["HF_TOKEN"] = token
    return token

from datasets import load_dataset, Dataset, DatasetDict
MODEL_ID = "google/gemma-3-270m-it"
DATASET_ID = "AmareshHebbar/clinical-summarizer-sft"
N_TRAIN_ROWS, N_EVAL_CASES, SEED = 1500, 10, 42
MAX_SEQUENCE_LENGTH = 4096
OUTPUT_ROOT = Path("prepared_clinical_dataset")
assert N_TRAIN_ROWS > 0 and N_EVAL_CASES > 0 and MAX_SEQUENCE_LENGTH > 0

token = read_token()
api = HfApi(token=token)
previous = next((json.loads(p.read_text(encoding="utf-8")) for p in (
    Path("baseline_evaluation_diagnostic_10/generation_config.json"),
    Path("baseline_evaluation/generation_config.json")) if p.exists()), {})
assert previous.get("model", MODEL_ID) == MODEL_ID
model_revision = previous.get("model_revision") or api.model_info(MODEL_ID).sha
dataset_revision = previous.get("dataset_revision") or api.dataset_info(DATASET_ID).sha
print("Loading tokenizer only...", flush=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=model_revision, token=token, use_fast=True)
assert tokenizer.is_fast
print("Loading source training and validation splits...", flush=True)
raw_train = load_dataset(DATASET_ID, revision=dataset_revision, split=f"train[:{N_TRAIN_ROWS}]", token=token)
raw_eval = load_dataset(DATASET_ID, revision=dataset_revision, split="validation", token=token)


Enter your Hugging Face token in the VS Code input prompt.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading tokenizer only...
Loading source training and validation splits...


In [3]:
TOP_LEVEL_KEYS = (
    "visit motivation", "admission", "patient information", "patient medical history",
    "surgeries", "symptoms", "medical examinations", "diagnosis tests", "treatments", "discharge")
FORMAT_INSTRUCTION = (
    "Extract the clinical facts stated in the note into one JSON object. "
    "Return JSON only, without markdown or commentary. Use these top-level fields: "
    + ", ".join(TOP_LEVEL_KEYS) + ". "
    "Use the string None only for information absent from the note. "
    "Do not invent facts or repeat identical entries.")

def no_duplicate_keys(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError("Duplicate JSON key: " + key)
        result[key] = value
    return result

def reject_constant(value):
    raise ValueError("Nonstandard JSON constant: " + value)

def load_object(text, strict=True):
    obj = json.loads(text, strict=strict, object_pairs_hook=no_duplicate_keys,
                     parse_constant=reject_constant)
    if not isinstance(obj, dict):
        raise ValueError("Expected a JSON object")
    return obj

def has_nonmissing_value(value):
    if isinstance(value, dict):
        return any(has_nonmissing_value(v) for v in value.values())
    if isinstance(value, list):
        return any(has_nonmissing_value(v) for v in value)
    return value is not None and " ".join(str(value).casefold().split()) not in (
        "", "none", "null", "unknown", "not provided")

def clean_reference(raw):
    text = raw.strip()
    fenced = text.startswith("```") and text.endswith("```")
    if fenced:
        text = re.sub(r"^```(?:json)?\s*", "", text, count=1, flags=re.IGNORECASE)[:-3].strip()
    try:
        obj = load_object(text)
        repair = "removed_markdown_fence" if fenced else "none"
    except ValueError:
        # Only unescaped control characters are tolerated; serialization escapes them.
        # No guessed commas, missing braces, invented values, or LLM repair.
        obj = load_object(text, strict=False)
        repair = "escaped_control_characters" + ("_and_removed_fence" if fenced else "")
    if set(obj) != set(TOP_LEVEL_KEYS):
        raise ValueError("Unexpected top-level schema")
    if not has_nonmissing_value(obj):
        raise ValueError("Reference contains no nonmissing values")
    canonical = json.dumps(obj, ensure_ascii=False, separators=(",", ":"))
    assert load_object(canonical) == obj
    return canonical, repair

# Conservative repairs preserve values; ambiguous/broken syntax must be rejected.
mock = {key: "None" for key in TOP_LEVEL_KEYS}
mock["visit motivation"] = "pain"
assert clean_reference(json.dumps(mock))[1] == "none"
broken_control = json.dumps(mock).replace('"visit motivation": "pain"', '"visit motivation": "line1\nline2"')
assert clean_reference(broken_control)[1] == "escaped_control_characters"
assert json.loads(clean_reference(broken_control)[0])["visit motivation"] == "line1\nline2"
for invalid in ('{"x": 1,}', '{"x":1,"x":2}', '{"x":NaN}', '[]'):
    try:
        clean_reference(invalid)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid reference accepted")
print("Cleaning self-checks passed.")


Cleaning self-checks passed.


In [4]:
def prepare_row(example, split, index):
    messages = example["messages"]
    if [m["role"] for m in messages] not in (
        ["system", "user", "assistant"], ["system", "user", "model"]):
        raise ValueError("Expected a single system/user/assistant conversation")
    system, note, raw_answer = [m["content"] for m in messages]
    if not all(isinstance(x, str) and x.strip() for x in (system, note, raw_answer)):
        raise ValueError("Empty or nontext conversation content")
    answer, repair = clean_reference(raw_answer)
    instruction = system.strip() + "\n\n" + FORMAT_INSTRUCTION
    user = [{"role": "user", "content": instruction + "\n\n" + note}]
    prompt = tokenizer.apply_chat_template(user, tokenize=False, add_generation_prompt=True)
    text = tokenizer.apply_chat_template(user + [{"role": "assistant", "content": answer}], tokenize=False)
    if not text.startswith(prompt + answer):
        raise RuntimeError("Unexpected tokenizer template boundary")
    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    positions = [i for i,(a,b) in enumerate(enc["offset_mapping"]) if b > len(prompt) and b > a]
    if not positions or enc["offset_mapping"][positions[0]][0] < len(prompt):
        raise RuntimeError("Assistant boundary shares a prompt token; inspect template")
    first_answer = positions[0]
    ids = enc["input_ids"]
    if len(ids) > MAX_SEQUENCE_LENGTH:
        raise ValueError(f"Complete sequence too long: {len(ids)}")
    labels = [-100] * first_answer + ids[first_answer:]
    assert len(labels) == len(ids) and any(v != -100 for v in labels[1:])
    return {"case_id": f"{split}:{index}", "source_split": split, "source_index": index,
            "source_sha256": digest(messages),
            "note_sha256": digest(" ".join(note.casefold().split())),
            "repair": repair, "system": instruction, "note": note, "reference": answer,
            "messages": [{"role": "system", "content": instruction},
                         {"role": "user", "content": note},
                         {"role": "assistant", "content": answer}],
            "prompt_text": prompt, "text": text,
            "input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels,
            "prompt_tokens": first_answer, "total_tokens": len(ids),
            "supervised_tokens": len(ids) - first_answer}

logs = []
def prepare_split(raw, name):
    result, seen = [], set()
    for i, example in enumerate(raw):
        try:
            row = prepare_row(example, name, i)
            if row["note_sha256"] in seen:
                raise ValueError("Duplicate normalized note within split")
            seen.add(row["note_sha256"])
            result.append(row)
            logs.append({"split": name, "source_index": i, "status": "accepted", "detail": row["repair"]})
        except (ValueError, KeyError, TypeError) as exc:
            logs.append({"split": name, "source_index": i, "status": "rejected", "detail": str(exc)})
        if (i+1) % 100 == 0:
            print(f"{name}: {i+1}/{len(raw)} checked; {len(result)} eligible", flush=True)
    return result

train_rows = prepare_split(raw_train, "train")
eval_pool = prepare_split(raw_eval, "validation")
# Reserve the complete eligible validation pool, not only the sampled evaluation cases.
eval_hashes = {r["note_sha256"] for r in eval_pool}
for row in train_rows:
    if row["note_sha256"] in eval_hashes:
        logs.append({"split": "train", "source_index": row["source_index"],
                     "status": "excluded_cross_split_duplicate", "detail": "Note appears in validation"})
train_rows = [r for r in train_rows if r["note_sha256"] not in eval_hashes]
assert train_rows and len(eval_pool) >= N_EVAL_CASES, "Not enough eligible examples; inspect rejection logs."
eval_rows = random.Random(SEED).sample(eval_pool, N_EVAL_CASES)
assert not ({r["note_sha256"] for r in train_rows} & {r["note_sha256"] for r in eval_rows})
prepared = DatasetDict({"train": Dataset.from_list(train_rows), "validation": Dataset.from_list(eval_rows)})
print(prepared)


train: 100/1500 checked; 100 eligible
train: 200/1500 checked; 199 eligible
train: 300/1500 checked; 296 eligible
train: 400/1500 checked; 392 eligible
train: 500/1500 checked; 489 eligible
train: 600/1500 checked; 585 eligible
train: 700/1500 checked; 681 eligible
train: 800/1500 checked; 778 eligible
train: 900/1500 checked; 875 eligible
train: 1000/1500 checked; 970 eligible
train: 1100/1500 checked; 1063 eligible
train: 1200/1500 checked; 1157 eligible
train: 1300/1500 checked; 1249 eligible
train: 1400/1500 checked; 1341 eligible
train: 1500/1500 checked; 1432 eligible
validation: 100/3000 checked; 97 eligible
validation: 200/3000 checked; 195 eligible
validation: 300/3000 checked; 295 eligible
validation: 400/3000 checked; 392 eligible
validation: 500/3000 checked; 491 eligible
validation: 600/3000 checked; 588 eligible
validation: 700/3000 checked; 685 eligible
validation: 800/3000 checked; 783 eligible
validation: 900/3000 checked; 880 eligible
validation: 1000/3000 checked; 97

In [5]:
config = {"schema_version": 1, "model": MODEL_ID, "model_revision": model_revision,
          "dataset": DATASET_ID, "dataset_revision": dataset_revision,
          "candidate_train_rows": N_TRAIN_ROWS, "eval_cases": N_EVAL_CASES, "seed": SEED,
          "max_sequence_length": MAX_SEQUENCE_LENGTH, "format_instruction": FORMAT_INSTRUCTION,
          "transformers": importlib.metadata.version("transformers"),
          "datasets": importlib.metadata.version("datasets"),
          "tokenizer_add_special_tokens": False, "loss": "assistant and closing tokens only"}
content_hashes = {split: digest(prepared[split].to_list()) for split in prepared}
manifest = {"config": config, "split_sha256": content_hashes,
            "counts": {split: len(prepared[split]) for split in prepared},
            "eligible_validation_pool": len(eval_pool),
            "repair_counts_saved": dict(Counter(r["repair"] for r in train_rows + eval_rows)),
            "log_status_counts": dict(Counter(r["status"] for r in logs)),
            "lengths": {split: {"max": max(prepared[split]["total_tokens"]),
                                  "mean": sum(prepared[split]["total_tokens"])/len(prepared[split])}
                        for split in prepared}}
version = digest({"config": config, "splits": content_hashes})[:16]
output_dir = OUTPUT_ROOT / version
if output_dir.exists():
    raise FileExistsError(f"{output_dir} already exists; preserve it or choose another OUTPUT_ROOT.")
output_dir.mkdir(parents=True)
prepared.save_to_disk(str(output_dir / "dataset"))
for split in prepared:
    prepared[split].to_json(str(output_dir / f"{split}.jsonl"), force_ascii=False)
pd.DataFrame(logs).to_csv(output_dir / "preparation_log.csv", index=False)
(output_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
# A small pointer lets subsequent notebooks find the most recently prepared version.
(OUTPUT_ROOT / "latest.json").write_text(json.dumps({"version": version}, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))
print("Prepared dataset:", output_dir.resolve())
print("Training must consume saved input_ids/attention_mask/labels without truncating or reformatting.")


Saving the dataset (0/1 shards):   0%|          | 0/1325 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/10 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

{
  "config": {
    "schema_version": 1,
    "model": "google/gemma-3-270m-it",
    "model_revision": "ac82b4e820549b854eebf28ce6dedaf9fdfa17b3",
    "dataset": "AmareshHebbar/clinical-summarizer-sft",
    "dataset_revision": "4ca20e0a38cc9235045c973b5503d49952fd1fac",
    "candidate_train_rows": 1500,
    "eval_cases": 10,
    "seed": 42,
    "max_sequence_length": 4096,
    "format_instruction": "Extract the clinical facts stated in the note into one JSON object. Return JSON only, without markdown or commentary. Use these top-level fields: visit motivation, admission, patient information, patient medical history, surgeries, symptoms, medical examinations, diagnosis tests, treatments, discharge. Use the string None only for information absent from the note. Do not invent facts or repeat identical entries.",
    "transformers": "4.57.6",
    "datasets": "4.8.5",
    "tokenizer_add_special_tokens": false,
    "loss": "assistant and closing tokens only"
  },
  "split_sha256": {
    "trai

In [6]:
import shutil
archive = shutil.make_archive(str(OUTPUT_ROOT.resolve()), "zip", root_dir=OUTPUT_ROOT.parent,
                             base_dir=OUTPUT_ROOT.name)
print("Download and preserve:", archive)
# Browser Colab only: from google.colab import files; files.download(archive)


Download and preserve: /content/prepared_clinical_dataset.zip
